Imports

In [ ]:
import os
from glob import glob
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
import MinkowskiEngine as ME
import numpy as np
import open3d as o3d
import cv2
import random
import matplotlib.pyplot as plt
import gc
from copy import deepcopy

Constants and hyperparameters

In [ ]:
SEQUENCE_LENGTH = 100
NUM_EPOCHS = 50
TRUNC=20
LR, LRD, LRD_STEP = 0.002, 1 , 1
LAMBDA_BOX, LAMBDA_REG, LAMBDA_CAR, LAMBDA_REP, LAMBDA_RAY, LAMBDA_BCE = 1, 1, 0, 0.1, 10, 10
BOX_SIZE, REP_R, PW = 2, 2, 50
ALPHA=0.2

Camera and map properties

In [ ]:
#Intrinsics
FX, FY, cx, cy = 382.7142944335938, 382.7142944335938, 316.7781372070312, 241.8565368652344 

#Extrinsics
transforms={
    "event_to_rgb": np.linalg.inv(
   [[0.99851105429082, -0.011029738783062, 0.0534230223912167, 0.019354],
    [0.0109594874100519, 0.999938650448127, 0.00160778595796204, -0.048871],
    [-0.0534374783718687, -0.0010199031106552, 0.998570676368429, -0.06149],
    [0.0, 0.0, 0.0, 1.0]]),
    "lidar_to_event": 
   [[0.0708534541759194, -0.997473033859274, 0.00522826500049706, -0.00446292653069122],
    [0.00519730032699972, -0.00487219752859147, -0.99997462455832, -0.114648648633838],
    [0.997473195680291, 0.0708788291016217, 0.00483895490072228, -0.128128003487762],
    [0.0, 0.0, 0.0, 1.0]],
    "depth_to_rgb":
   [[0.999986115848177, 0.00201034610955854, 0.00487099778244396, -0.059056371],
    [-0.00202387960198555, 0.999994101493743, 0.00277504758125118, 0.00020030836],
    [-0.00486539024472426, -0.00278486736512245, 0.9999842861223, 0.00059907947],
    [0.0, 0.0, 0.0, 1.0]],
    "rgb_to_robot": np.linalg.inv(
   [[0.0, -1.0, 0.0, -0.012],
    [0.0, 0.0, -1.0, 0.132],
    [1.0, 0.0, 0.0, -0.1],
    [0.0, 0.0, 0.0, 1.0]]),
    "marker_to_rgb": 
   [[0.999850004761365,   0.0140076835197648,   0.0101859109120805,  0.00134740633787912],
    [0.0139197148362179,   -0.999865644059574,  0.00865652171554533,  -0.0060681888761388],
    [0.0103058001910142, -0.00851343830326158,   -0.999910651933802, -0.00717831706165138],
    [0.0, 0.0, 0.0, 1.0]] 
}

T_depth_to_robot=transforms["rgb_to_robot"] @ transforms["depth_to_rgb"]
T_lidar_to_robot=transforms["rgb_to_robot"] @ transforms["event_to_rgb"] @ transforms["lidar_to_event"]

#Map
VOXEL_SIZE = 0.1
GRID_SIZE = 64
MAP_SIZE = GRID_SIZE * VOXEL_SIZE

Model Architecture

In [ ]:
class SparseUNet4D(nn.Module):
    def __init__(self, prune_alpha=ALPHA):
        super().__init__()
        self.alpha = prune_alpha
        CHANNELS = [16, 32, 64, 128, 256]
        
        self.enc1 = nn.Sequential(
            ME.MinkowskiConvolution(3, CHANNELS[0], kernel_size=3, stride=1, dimension=4), 
            ME.MinkowskiBatchNorm(CHANNELS[0]),
            ME.MinkowskiReLU()
        )
        self.enc2 = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[0], CHANNELS[1], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[1]),
            ME.MinkowskiReLU()
        )
        self.enc3 = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[1], CHANNELS[2], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[2]),
            ME.MinkowskiReLU()
        )
        self.enc4 = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[2], CHANNELS[3], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[3]),
            ME.MinkowskiReLU()
        )
        self.enc5 = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[3], CHANNELS[4], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[4]),
            ME.MinkowskiReLU()
        )
###decoders (5 decoders)
        self.dec4_t = nn.Sequential(
            ME.MinkowskiConvolutionTranspose(CHANNELS[4], CHANNELS[3], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[3]),
            ME.MinkowskiReLU()
        )
        self.dec4_c = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[3] * 2, CHANNELS[3], kernel_size=3, stride=1, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[3]),
            ME.MinkowskiReLU()
        )
        self.dec3_t = nn.Sequential(
            ME.MinkowskiConvolutionTranspose(CHANNELS[3], CHANNELS[2], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[2]),
            ME.MinkowskiReLU()
        )
        self.dec3_c = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[2] * 2, CHANNELS[2], kernel_size=3, stride=1, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[2]),
            ME.MinkowskiReLU()
        )
        self.dec2_t = nn.Sequential(
            ME.MinkowskiConvolutionTranspose(CHANNELS[2], CHANNELS[1], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[1]),
            ME.MinkowskiReLU()
        )
        self.dec2_c = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[1] * 2, CHANNELS[1], kernel_size=3, stride=1, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[1]),
            ME.MinkowskiReLU()
        )
        self.dec1_t = nn.Sequential(
            ME.MinkowskiConvolutionTranspose(CHANNELS[1], CHANNELS[0], kernel_size=3, stride=2, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[0]),
            ME.MinkowskiReLU()
        )
        self.dec1_c = nn.Sequential(
            ME.MinkowskiConvolution(CHANNELS[0] * 2, CHANNELS[0], kernel_size=3, stride=1, dimension=4),
            ME.MinkowskiBatchNorm(CHANNELS[0]),
            ME.MinkowskiReLU()
        )

        self.prune4 = ME.MinkowskiConvolution(CHANNELS[3], 1, kernel_size=1, dimension=4)
        self.prune3 = ME.MinkowskiConvolution(CHANNELS[2], 1, kernel_size=1, dimension=4)
        self.prune2 = ME.MinkowskiConvolution(CHANNELS[1], 1, kernel_size=1, dimension=4)
        self.prune1 = ME.MinkowskiConvolution(CHANNELS[0], 1, kernel_size=1, dimension=4)

        self.out_conv = ME.MinkowskiConvolution(CHANNELS[0], 3, kernel_size=1, dimension=4)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)

        d4_t = self.dec4_t(e5)
        d4 = ME.cat(d4_t, e4)
        d4 = self.dec4_c(d4)
        prune_scores4 = self.prune4(d4)

        d3_t = self.dec3_t(d4)
        d3 = ME.cat(d3_t, e3)
        d3 = self.dec3_c(d3)
        prune_scores3 = self.prune3(d3)

        d2_t = self.dec2_t(d3)
        d2 = ME.cat(d2_t, e2)
        d2 = self.dec2_c(d2)
        prune_scores2 = self.prune2(d2)

        d1_t = self.dec1_t(d2)
        d1 = ME.cat(d1_t, e1)
        d1 = self.dec1_c(d1)
        prune_scores1 = self.prune1(d1)

        out = self.out_conv(d1)

        for o in [out]:
            o.F.sigmoid_()
        return out, [prune_scores1, prune_scores2, prune_scores3, prune_scores4]

def suppress_by_pruning(output_map: ME.SparseTensor,
                        pruning_map: ME.SparseTensor,
                        alpha: float) -> ME.SparseTensor | None:
    if output_map is None or output_map.C.numel() == 0:
        return None
    if pruning_map is None or pruning_map.C.numel() == 0:
        return None

    outC = output_map.C.to(torch.int64)
    prC  = pruning_map.C.to(torch.int64)

    def pack5(C):
        b, x, y, z, t = C[:,0], C[:,1]+GRID_SIZE // 2, C[:,2]+GRID_SIZE // 2, C[:,3]+GRID_SIZE // 2, C[:,4]
        return (((((b << 6) + t) << 18) + x) << 18 | y) << 18 | z

    outK = pack5(outC)
    prK  = pack5(prC)

    prK_sorted, order = torch.sort(prK)
    scores_sorted = torch.sigmoid(pruning_map.F.squeeze(-1)[order])

    idx = torch.searchsorted(prK_sorted, outK)
    idx = torch.clamp(idx, 0, prK_sorted.numel() - 1)

    matched = (prK_sorted[idx] == outK)
    keep = matched & (scores_sorted[idx] >= float(alpha))

    if keep.sum().item() == 0:
        return None

    return ME.MinkowskiPruning()(output_map, keep)

device = torch.device("cuda")

Loss functions

In [ ]:
def BCE(prune_scores: ME.SparseTensor, gt_occ: ME.SparseTensor, pw:int):
    device, logits = prune_scores.F.device, prune_scores.F.view(-1) 
    occ = torch.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), device=device, dtype=torch.bool)

    gt_xyz = gt_occ.C[:, 1:4].to(torch.int64) + GRID_SIZE // 2
    occ[gt_xyz[:,0], gt_xyz[:,1], gt_xyz[:,2]] = True

    pr_xyz = prune_scores.C[:, 1:4].to(torch.int64) + GRID_SIZE // 2
    labels = occ[pr_xyz[:,0], pr_xyz[:,1], pr_xyz[:,2]].float()

    P=labels.sum(); N=labels.numel()-P

    return F.binary_cross_entropy_with_logits(logits, labels, pos_weight=(N / (P + 1e-6)).clamp(1.0,pw))


def repulsion_loss(pred: ME.SparseTensor, radius: float, max_pred: int = 1024, eps: float = 1e-9):
    device = pred.F.device
    C = pred.C[:, 1:4].to(torch.float32)
    off = pred.F[:, :3]
    pts = (C + off) * VOXEL_SIZE
    N = pts.size(0)
    if N <= 1:
        return torch.zeros((), device=device)

    if N > max_pred:
        idx = torch.randperm(N, device=device)[:max_pred]
        pts = pts[idx]

    r2 = float(radius * radius)
    diff = pts[:, None, :] - pts[None, :, :]
    dist2 = (diff * diff).sum(dim=-1) + eps

    mask = ~torch.eye(pts.size(0), dtype=torch.bool, device=device)
    dist2 = dist2[mask]

    rep = torch.clamp(r2 - dist2, min=0.0)
    return (rep ** 2).mean()


def chamfer_distance(gt_pts:torch.Tensor, pred_pts:torch.Tensor, max_gt=1024, max_pred:int=1024):
    M,N=gt_pts.size(0),pred_pts.size(0)
    if N==0 or M==0:
        return torch.tensor(0.0,device=gt_pts.device)
    if M>max_gt:
        gt_pts=gt_pts[torch.randperm(M, device=gt_pts.device)[:max_gt]]
    if N>max_pred:
        pred_pts=pred_pts[torch.randperm(N, device=pred_pts.device)[:max_pred]]
    diff=gt_pts[:,None,:]-pred_pts[None,:,:]
    dist1=(diff**2).sum(dim=-1)
    min_dist1, _ = dist1.min(dim=1)
    dist2=(diff**2).sum(dim=0)
    min_dist2, _ = dist2.min(dim=1)
    return min_dist1.mean()

def box_loss(pred:ME.SparseTensor, gt:ME.SparseTensor, box_size:int, device: torch.device, chamfer_weight:float=1.0): 
    predC_int, gtC_int = pred.C[:,1:4].cpu(), gt.C[:,1:4].cpu() 
    predC, gtC = pred.C[:,1:4].to(device=device, dtype=torch.float32), gt.C[:,1:4].to(device=device, dtype=torch.float32) 
    predF, gtF = pred.F, gt.F 
    pred_pts, gt_pts = (predC.float() + predF) * VOXEL_SIZE, (gtC.float() + gtF) * VOXEL_SIZE 
    pred_map = {tuple(c.tolist()): i for i, c in enumerate(predC_int)} #hashing of occupied voxels 
    local_losses, chamfer_gt_list = [], [] 
    
    for m in range(len(gtC_int)): 
        gx,gy,gz = gtC_int[m].tolist() 
        gt_point = gt_pts[m] 
        neighbor_idx=[] 
        for dx in range(-box_size,box_size+1): 
            for dy in range(-box_size, box_size+1): 
                for dz in range(-box_size, box_size+1): 
                    coords=(gx+dx,gy+dy,gz+dz) 
                    if coords in pred_map: 
                        neighbor_idx.append(pred_map[coords]) 
                        if neighbor_idx: 
                            idxs = torch.tensor(neighbor_idx, device=device, dtype=torch.long) 
                            pred_centroid=pred_pts[idxs].mean(0) 
                            local_losses.append(((pred_centroid-gt_point)**2).sum()) 
                        else: 
                            chamfer_gt_list.append(gt_point) 
        if local_losses: 
            local_losses=torch.stack(local_losses).mean() 
        else: 
            local_losses=torch.tensor(0.0,device=device)

        chamfer_loss=torch.tensor(0.0, device=device) 
        if len(chamfer_gt_list)>0: 
            gt_far=torch.stack(chamfer_gt_list, dim=0) 
        chamfer_loss=chamfer_distance(gt_far,pred_pts)
        return local_losses+chamfer_weight*chamfer_loss


def box_loss_gpu(pred:ME.SparseTensor, gt:ME.SparseTensor, box_size:int, device: torch.device, chamfer_weight:float=1.0):
    if pred is None or gt is None or pred.C.numel() == 0 or gt.C.numel()==0:
        return torch.zeros((),device=device)
    predC, gtC = pred.C[:,1:4].to(device=device, dtype=torch.int64), gt.C[:,1:4].to(device=device, dtype=torch.int64)
    predF, gtF = pred.F[:,:3], gt.F[:,:3]
    pred_pts, gt_pts = (predC.float() + predF) * VOXEL_SIZE, (gtC.float() + gtF) * VOXEL_SIZE

    predC, gtC = predC+GRID_SIZE//2 , gtC+GRID_SIZE//2
    def in_bounds(ijk):
        return ((ijk[:,0]>=0) & (ijk[:,0]<=GRID_SIZE-1) &
                (ijk[:,1]>=0) & (ijk[:,1]<=GRID_SIZE-1) &
                (ijk[:,2]>=0) & (ijk[:,2]<=GRID_SIZE-1))
    vpred, vgt = in_bounds(predC), in_bounds(gtC)
    if vpred.sum().item()==0 or vgt.sum().item()==0:
        return torch.zeros((),device=device)
    predC, pred_pts = predC[vpred], pred_pts[vpred]
    gtC, gt_pts = gtC[vgt], gt_pts[vgt]

    index_grid = torch.full((GRID_SIZE,GRID_SIZE,GRID_SIZE),-1,device=device,dtype=torch.int32) # Empty grid
    index_grid[predC[:,0],predC[:,1],predC[:,2]]=torch.arange(predC.size(0),device=device,dtype=torch.int32) # M pred pts are inserted into empty grid for O(1) lookup
    offsets = torch.stack(torch.meshgrid(
        torch.arange(-box_size,box_size+1,device=device),
        torch.arange(-box_size,box_size+1,device=device),
        torch.arange(-box_size,box_size+1,device=device), indexing="ij"),dim=-1).reshape(-1,3) # K*3 grid of neighbor offsets
    neighbor = (gtC[:,None,:]+offsets[None,:,:]).clamp(0,GRID_SIZE-1) # M*K*3, all K neighbors for each of the M pred pts
    m=((neighbor[...,0]>=0) & (neighbor[...,0]<=GRID_SIZE-1) & 
       (neighbor[...,1]>=0) & (neighbor[...,1]<=GRID_SIZE-1) & 
       (neighbor[...,2]>=0) & (neighbor[...,2]<=GRID_SIZE-1)) # M*K mask to make sure neighbors are inside grid (can be accessed by index w/o error)
    idx = index_grid[neighbor[...,0], neighbor[...,1], neighbor[...,2]] # M*K grid that contains either -1 or the index of the neighbor in predC
    idx = idx.masked_fill(~m,-1) # filter invalid neighbors to avoid indexing errors

    has = (idx>=0) # M*K boolean matrix
    has_any=has.any(dim=1) # M, check for each gt voxel if it has a pred pt in its neighborhood
    local_loss, chamfer_loss = torch.zeros((),device=device), torch.zeros((),device=device)
    if has_any.any():
        idx=idx.clamp_min(0).to(torch.long)
        gathered = pred_pts[idx] * has.unsqueeze(-1) # M*K*3 grid containing all valid 3D points for each gt point
        count = has.sum(dim=1).clamp_min(1).to(torch.float32) # M, nb of neighbors per gt
        centroid = gathered.sum(dim=1) / count.unsqueeze(-1) # M*3 centroids of boxes
        local_loss = (centroid[has_any]-gt_pts[has_any]).pow(2).sum(dim=1).sum()
    '''if (~has_any).any():
        chamfer_loss = chamfer_distance(gt_pts[~has_any], pred_pts)'''
    return local_loss


def sparsity_regularizer(pruning_scores, device):
    s = torch.zeros((), device=device)
    for ps in pruning_scores:
        s = s + ps.F.mean()
    return s

def cardinality_loss(pruning_scores, gt,ratio=1.0, eps=1e-6):
    N_gt=gt.C.shape[0]
    prob=torch.sigmoid(pruning_scores[0].F).view(-1)
    N_pred_soft=prob.sum()
    target=torch.tensor(ratio*float(N_gt),device=device)
    return ((N_pred_soft - target) / (target + eps))**2

def inject_voxels(pruning_scores,eps=1e-6):
    prob=(pruning_scores[0].F).view(-1)
    return 1/(prob.sum()+eps)


@torch.no_grad()
def build_rmin_map_from_occ_vox(gt_occ_vox: ME.SparseTensor,
                               T_robot_to_lidar: torch.Tensor,
                               voxel_size: float,
                               az_bins: int = 360,
                               el_bins: int = 64):
    """
    Returns:
      rmin_map:  [el_bins, az_bins] float32 (inf where empty)
      valid_map: [el_bins, az_bins] bool
    """
    device = gt_occ_vox.F.device
    T = torch.from_numpy(T_robot_to_lidar.astype(np.float32)).to(device=device)

    C = gt_occ_vox.C[:, 1:4].to(torch.float32)    # voxel indices in robot grid
    off = gt_occ_vox.F[:, :3].to(torch.float32)   # GT offsets (already in [0,1) from your prepare_voxels)
    pts_robot = (C + off) * voxel_size            # [N,3] robot frame points

    ones = torch.ones((pts_robot.size(0), 1), device=device, dtype=torch.float32)
    pts_h = torch.cat([pts_robot, ones], dim=1)
    pts_lidar = (T @ pts_h.T).T[:, :3]            # [N,3] lidar frame

    x, y, z = pts_lidar[:, 0], pts_lidar[:, 1], pts_lidar[:, 2]
    r = torch.sqrt(x*x + y*y + z*z).clamp_min(1e-6)

    az = torch.atan2(y, x)                                     # [-pi, pi]
    el = torch.asin((z / r).clamp(-1.0, 1.0))                  # [-pi/2, pi/2]

    iaz = torch.floor((az + torch.pi) * (az_bins / (2.0 * torch.pi))).to(torch.int64).clamp(0, az_bins - 1)
    iel = torch.floor((el + 0.5 * torch.pi) * (el_bins / torch.pi)).to(torch.int64).clamp(0, el_bins - 1)

    flat_idx = iel * az_bins + iaz
    flat_size = az_bins * el_bins

    rmin = torch.full((flat_size,), float("inf"), device=device, dtype=torch.float32)
    rmin = rmin.scatter_reduce(0, flat_idx, r, reduce="amin", include_self=True)

    valid_i = torch.zeros((flat_size,), device=device, dtype=torch.int32)
    ones_i  = torch.ones_like(flat_idx, device=device, dtype=torch.int32)
    valid_i = valid_i.scatter_reduce(0, flat_idx, ones_i, reduce="amax", include_self=True)
    valid = valid_i > 0

    return rmin.view(el_bins, az_bins), valid.view(el_bins, az_bins)


def ray_loss(prune_scores: ME.SparseTensor,
                     gt_occ_vox: ME.SparseTensor,
                     T_robot_to_lidar,
                     az_bins: int = 720,
                     el_bins: int = 128,
                     free_margin_m: float = 0.10,
                     eps: float = 1e-6):
    """
    Ray loss for pruning probabilities.
    Positives: predicted voxels whose (x,y,z) voxel coord is occupied in gt_occ_vox
    Negatives: predicted voxels that are in front of first return along same (az,el)
    Ignore: everything else
    """
    device = prune_scores.F.device

    # --- predicted logits ---
    p = prune_scores.F
    if p.dim() == 2 and p.size(1) == 1:
        p = p[:, 0]
    else:
        p = p.view(-1)

    # --- rmin map from GT occupied voxels ---
    rmin_map, valid_map = build_rmin_map_from_occ_vox(
        gt_occ_vox=gt_occ_vox,
        T_robot_to_lidar=T_robot_to_lidar,
        voxel_size=VOXEL_SIZE,
        az_bins=az_bins,
        el_bins=el_bins
    )

    # --- positives via dense occupancy grid (no hashing) ---
    half = GRID_SIZE // 2
    gt_xyz = gt_occ_vox.C[:, 1:4].to(torch.int64) + half
    pr_xyz = prune_scores.C[:, 1:4].to(torch.int64) + half
    occ = torch.zeros((GRID_SIZE, GRID_SIZE, GRID_SIZE), device=device, dtype=torch.bool)

    valid_gt = (gt_xyz[:, 0].ge(0) & gt_xyz[:, 0].lt(GRID_SIZE) &
                gt_xyz[:, 1].ge(0) & gt_xyz[:, 1].lt(GRID_SIZE) &
                gt_xyz[:, 2].ge(0) & gt_xyz[:, 2].lt(GRID_SIZE))
    gt_xyz = gt_xyz[valid_gt]
    if gt_xyz.numel() > 0:
        occ[gt_xyz[:, 0], gt_xyz[:, 1], gt_xyz[:, 2]] = True

    valid_pr = (pr_xyz[:, 0].ge(0) & pr_xyz[:, 0].lt(GRID_SIZE) &
                pr_xyz[:, 1].ge(0) & pr_xyz[:, 1].lt(GRID_SIZE) &
                pr_xyz[:, 2].ge(0) & pr_xyz[:, 2].lt(GRID_SIZE))
    pos = torch.zeros((p.numel(),), device=device, dtype=torch.bool)
    pos[valid_pr] = occ[pr_xyz[valid_pr, 0], pr_xyz[valid_pr, 1], pr_xyz[valid_pr, 2]]

    T = torch.from_numpy(T_robot_to_lidar.astype(np.float32)).to(device=device)
    pts_robot = (prune_scores.C[:, 1:4].to(torch.float32) + 0.5) * VOXEL_SIZE  # voxel center (pruning is per-voxel)
    ones = torch.ones((pts_robot.size(0), 1), device=device, dtype=torch.float32)
    pts_h = torch.cat([pts_robot, ones], dim=1)
    pts_lidar = (T @ pts_h.T).T[:, :3]

    x, y, z = pts_lidar[:, 0], pts_lidar[:, 1], pts_lidar[:, 2]
    r = torch.sqrt(x*x + y*y + z*z).clamp_min(1e-6)
    az = torch.atan2(y, x)
    el = torch.asin((z / r).clamp(-1.0, 1.0))

    iaz = torch.floor((az + torch.pi) * (az_bins / (2.0 * torch.pi))).to(torch.int64).clamp(0, az_bins - 1)
    iel = torch.floor((el + 0.5 * torch.pi) * (el_bins / torch.pi)).to(torch.int64).clamp(0, el_bins - 1)
    rmin = rmin_map[iel, iaz]
    valid = valid_map[iel, iaz]

    # free space labels should NEVER override positives
    free = valid & (r < (rmin - free_margin_m)) & (~pos)
    labels = torch.full((p.numel(),), -1.0, device=device, dtype=torch.float32)
    labels[pos] = 1.0
    labels[free] = 0.0

    use = labels >= 0.0
    if use.sum().item() == 0:
        return torch.zeros((), device=device)

    return F.binary_cross_entropy_with_logits(p[use], labels[use])

def refined_loss(prun, pred, gt, gt_p,Trl, device,
                 lambda_car, 
                 lambda_box, box_size, 
                 lambda_reg,
                 lambda_rep, rep_r,
                 lambda_bce, pw,
                 lambda_ray):
    return lambda_bce*BCE(prun[0],gt,pw) + lambda_box*box_loss_gpu(pred,gt,box_size,device)

File loading

In [ ]:
DEPTH_DIR1 = "/home/daniel/hm_ws/bet_bldg/depth_2"
LIDAR_DIR1 = "/home/daniel/hm_ws/bet_bldg/gt_npy2"
POSE_DIR1 = "/home/daniel/hm_ws/bet_bldg/poses"
DEPTH_DIR2 = "/home/daniel/hm_ws/ar_bldg/depth_2"
LIDAR_DIR2 = "/home/daniel/hm_ws/ar_bldg/gt_npy2"
POSE_DIR2 = "/home/daniel/hm_ws/ar_bldg/poses"
DEPTH_DIR3 = "/home/daniel/hm_ws/class/depth_2"
LIDAR_DIR3 = "/home/daniel/hm_ws/class/gt_npy2"
POSE_DIR3 = "/home/daniel/hm_ws/class/poses"
SYNTH_CAM1 =  "/home/daniel/hm_ws/synthetic1/voxels_yaw"
SYNTH_GT1 = "/home/daniel/hm_ws/synthetic1/world_mesh.obj"
SYNTH_POSE1 = "/home/daniel/hm_ws/synthetic1/poses"

def _ts_key(path: str):
    return int(os.path.splitext(os.path.basename(path))[0])

def generate_sequence(sequence_length,depth_dirs,gt_dirs,pose_dirs,seed=None):
    assert len(depth_dirs)==len(gt_dirs)==len(pose_dirs), "Need one to one correspondance between paths"
    depth_sequences,gt_sequences,pose_sequences= [],[],[]

    for ddir, gdir, pdir in zip(depth_dirs, gt_dirs, pose_dirs):
        d_files = sorted(glob(os.path.join(ddir, "*.txt")),key=_ts_key)
        p_files = sorted(glob(os.path.join(pdir,"*.npy")),key=_ts_key)
        assert len(d_files) == len(p_files), f"Mismatch in {ddir} vs {pdir}"

        for i in range(len(d_files)//sequence_length):
            start, end = i*sequence_length, min((i+1)*sequence_length,len(d_files))
            depth_sequences.append(d_files[start:end])
            pose_sequences.append(p_files[start:end])
            gt_sequences.append(gdir)

    '''trio = list(zip(depth_sequences, gt_sequences,pose_sequences))
    rng = np.random.default_rng(seed)
    rng.shuffle(trio)
    depth_sequences, gt_sequences, pose_sequences = zip(*trio)'''
    return depth_sequences, gt_sequences, pose_sequences  

depth_sequences, gt_sequences, pose_sequences = generate_sequence(SEQUENCE_LENGTH,[SYNTH_CAM1],[SYNTH_GT1],[SYNTH_POSE1])
nb_seq=len(depth_sequences)
depth_train, depth_val, depth_test = depth_sequences[:int(0.85*nb_seq)], depth_sequences[int(0.85*nb_seq):], []
gt_train, gt_val, gt_test = gt_sequences[:int(0.85*nb_seq)], gt_sequences[int(0.85*nb_seq):], []
pose_train, pose_val, pose_test = pose_sequences[:int(0.85*nb_seq)], pose_sequences[int(0.85*nb_seq):], []
print(f"Train, Val, Test: {len(depth_train)}, {len(depth_val)}, {len(depth_test)}")
print(f"Train Sequence length: {len(depth_train[0])}, Val Sequence length {len(pose_val[0])}")

File processing

In [ ]:
#Helper functions
def png_to_npy(path):
    return cv2.imread(path,cv2.IMREAD_ANYDEPTH).astype(np.float32)

def shuffle(list1,list2,list3):
    c=list(zip(list1,list2,list3))
    random.shuffle(c)
    l1, l2, l3 = map(list, zip(*c))
    return l1, l2, l3

def distance_cap(cloud,d):
    mask = (
    (cloud[:,0] >= -d) & (cloud[:,0] <= d) &
    (cloud[:,1] >= -d) & (cloud[:,1] <= d) &
    (cloud[:,2] >= -d) & (cloud[:,2] <= d)
    )
    return cloud[mask]

def depth_to_points(depth, fx, fy, cx, cy):
    h, w = depth.shape
    us, vs = np.meshgrid(np.arange(w), np.arange(h))
    valid=(depth>0) & (np.isfinite(depth))
    z = depth[valid]
    x = (us[valid] - cx) * z / fx
    y = (vs[valid] - cy) * z / fy
    return np.stack([x, y, z], axis=-1).reshape(-1, 3)

def prepare_voxels(points, voxel_size, time_idx=0):
    vox = np.floor(points / voxel_size).astype(int)
    uvox, inv = np.unique(vox,axis=0,return_inverse=True)
    cnt = np.bincount(inv)
    sx, sy, sz = np.bincount(inv, weights = points[:,0]), np.bincount(inv, weights = points[:,1]), np.bincount(inv, weights = points[:,2])
    cent = np.stack([sx,sy,sz],1) / cnt[:,None]
    offsets = (cent / voxel_size) - uvox.astype(np.float32)
    tcol = np.full((uvox.shape[0],1),time_idx,np.float32)
    coords, feats = np.hstack([uvox,tcol]).astype(np.float32), offsets.astype(np.float32)
    return coords, feats

def voxel_to_npy(voxel_map,squash=False):
    if voxel_map is None:
        return None
    coords = voxel_map.C[:,1:4]
    feats  = torch.sigmoid(voxel_map.F) if squash else voxel_map.F
    pts=(coords+feats)*VOXEL_SIZE
    return pts.detach().cpu().numpy()

def world_to_robot(P_world, T_world_from_robot):
    R = T_world_from_robot[:3,:3]
    t = T_world_from_robot[:3, 3]
    return (P_world - t) @ R

@torch.no_grad()
def coords_to_dense(xyz, grid=GRID_SIZE):
    occ = torch.zeros((grid,grid,grid), device=device, dtype=torch.bool)
    half = GRID_SIZE//2
    idx = xyz.C[:,1:4].to(torch.int64) + half
    occ[idx[:,0],idx[:,1],idx[:,2]]=True
    return occ

@torch.no_grad()
def ME_to_points(tensor:ME.SparseTensor):
    pts=(tensor.C[:,1:4].to(device=device, dtype=torch.float32)+tensor.F[:,:3].to(device=device, dtype=torch.float32))*VOXEL_SIZE
    return pts

In [ ]:
#Main function
def npy_to_voxel(data, s, time_idx=0):
    if data is None: return None
    if s=="depth" or s=="lidar":
        if s=="depth":
            data=depth_to_points(data,FX,FY,cx,cy)/1000.0
        data = np.hstack([data, np.ones((len(data), 1))])
        if s=="depth":
            data=((T_depth_to_robot @ data.T).T)[:,:3]
        elif s=="lidar":
            data=((T_lidar_to_robot @ data.T).T)[:,:3]
        data=distance_cap(data,GRID_SIZE*VOXEL_SIZE/2)
    else:
        data=distance_cap(data,GRID_SIZE*VOXEL_SIZE/2)
    d_v,d_f = prepare_voxels(data,VOXEL_SIZE,time_idx)
    if d_v is None:
        return None
    ME_data= ME.SparseTensor(
            features=torch.from_numpy(d_f).float().to(device),
            coordinates=ME.utils.batched_coordinates([torch.from_numpy(d_v).int()]).to(device))
    return ME_data

Visualization

In [ ]:
def to_homogeneous(pts):
    return np.hstack([pts, np.ones((len(pts), 1))])

def visualize_alignment(points_input, points_gt=None, nb_frames=1000):
    pcd_input = o3d.geometry.PointCloud()
    if points_input is not None:
        pcd_input.points = o3d.utility.Vector3dVector(points_input)
        pcd_input.paint_uniform_color([1, 0, 0])  # Red for input points

    #points_gt = (T_lidar_to_robot @ to_homogeneous(distance_cap(points_gt, GRID_SIZE * VOXEL_SIZE / 2)).T).T[:, :3]
    pcd_gt = o3d.geometry.PointCloud()
    if points_gt is not None:
        pcd_gt.points = o3d.utility.Vector3dVector(points_gt)
        pcd_gt.paint_uniform_color([0, 1, 0])  # Green for GT points

    axes = o3d.geometry.TriangleMesh.create_coordinate_frame(size=0.5, origin=[0, 0, 0])

    counter ={"i":0}
    def callback(vis):
        counter["i"]+=1
        if counter["i"]>nb_frames:
            vis.close()
        return False
    o3d.visualization.draw_geometries_with_animation_callback([pcd_input, pcd_gt,axes],callback)


Pose utilities

In [ ]:
def load_pose(path):
    return np.load(path,allow_pickle=True).astype(np.float64)

def switch_pose(T_prev,T_cur,xyz):
    T_prev_to_cur = np.linalg.inv(T_cur) @ (T_prev)
    xyz=to_homogeneous(xyz)
    xyz=T_prev_to_cur@(xyz.T)
    xyz=(xyz[:3]/xyz[3]).T
    return xyz

def warp_and_revoxelize(prev_voxels: ME.SparseTensor,
                        T_prev: np.ndarray,
                        T_cur: np.ndarray,
                        time_idx: int,
                        debug: bool = False,
                        squash_offsets: bool = False,
                        depth_input: ME.SparseTensor | None = None):
    """ 
    Build a new SparseTensor in the *current* frame from previous output_voxels. 
    Coords will be [batch, x, y, z, t] with t = time_idx.
    Output voxels after transform that coincide with input will be discarded.
    """
    if prev_voxels is None:
        return None
    C_prev = prev_voxels.C
    F_prev = prev_voxels.F
    device = F_prev.device

    N0 = int(C_prev.size(0))
    if N0 == 0:
        return None

    # 1) Get 3D points in past frame
    voxel_idx_prev = C_prev[:, 1:4].to(torch.float32)

    offsets_prev_raw = F_prev[:, :3]
    offsets_prev = torch.sigmoid(offsets_prev_raw) if squash_offsets else offsets_prev_raw
    pts_prev = (voxel_idx_prev + offsets_prev) * VOXEL_SIZE

    # 2) Pose transform prev->cur
    T_prev_t = torch.from_numpy(T_prev).to(device=device, dtype=torch.float32)
    T_cur_t  = torch.from_numpy(T_cur ).to(device=device, dtype=torch.float32)
    T_prev_to_cur = torch.linalg.inv(T_cur_t) @ T_prev_t

    ones = torch.ones((pts_prev.size(0), 1), device=device)
    pts_prev_h = torch.cat([pts_prev, ones], dim=1)
    pts_cur = (T_prev_to_cur @ pts_prev_h.T).T[:, :3]

    # 3) Voxel indices in current frame
    voxel_idx_cur_all = torch.floor(pts_cur/VOXEL_SIZE).to(torch.int32)

    # 4) Clamp
    half = GRID_SIZE // 2
    mask = (
        (voxel_idx_cur_all[:, 0] >= -half) & (voxel_idx_cur_all[:, 0] < half) &
        (voxel_idx_cur_all[:, 1] >= -half) & (voxel_idx_cur_all[:, 1] < half) &
        (voxel_idx_cur_all[:, 2] >= -half) & (voxel_idx_cur_all[:, 2] < half)
    )

    voxel_idx_cur = voxel_idx_cur_all[mask]
    pts_cur_kept  = pts_cur[mask]
    F_prev_kept   = F_prev[mask]
    if voxel_idx_cur.numel() == 0:
        return None

    # 5) Offsets in new voxel
    offsets_cur = (pts_cur_kept / VOXEL_SIZE) - voxel_idx_cur.to(torch.float32)
    feats = F_prev_kept.clone()
    feats[:, :3] = offsets_cur

    # 6) Build coords
    batch_col = C_prev[mask, 0:1].to(torch.int32)
    time_col  = torch.full((voxel_idx_cur.size(0), 1), int(time_idx),
                           dtype=torch.int32, device=device)
    coords = torch.cat([batch_col, voxel_idx_cur, time_col], dim=1).contiguous()
    feats  = feats.contiguous()

    return ME.SparseTensor(features=feats, coordinates=coords, device=device)

Map creation

In [ ]:
def union_unique(a: ME.SparseTensor | None, b: ME.SparseTensor | None):
    if a is None: return b
    if b is None: return a
    C = torch.cat([a.C, b.C], dim=0)
    F = torch.cat([a.F, b.F], dim=0)

    half = GRID_SIZE // 2
    C64 = C.to(torch.int64)
    b0,x,y,z,t = C64[:,0], C64[:,1]+half, C64[:,2]+half, C64[:,3]+half, C64[:,4]
    key = (((((b0 << 6) + t) << 18) + x) << 18 | y) << 18 | z

    k, order = torch.sort(key)
    C = C[order]; F = F[order]
    keep = torch.ones_like(k, dtype=torch.bool)
    keep[1:] = (k[1:] != k[:-1])
    return ME.SparseTensor(features=F[keep], coordinates=C[keep], device=a.F.device)

def filter_to_mask(src: ME.SparseTensor | None, mask: ME.SparseTensor | None):
    """
    Keep voxels in src whose (b,x,y,z,t) coords exist in mask.
    """
    if src is None or mask is None:
        return None
    if src.C.numel() == 0 or mask.C.numel() == 0:
        return None

    device = src.F.device
    srcC = src.C.to(torch.int64)
    mC   = mask.C.to(torch.int64)

    half = GRID_SIZE // 2

    def pack(C):
        b, x, y, z, t = C[:,0], C[:,1]+half, C[:,2]+half, C[:,3]+half, C[:,4]
        return (((((b << 6) + t) << 18) + x) << 18 | y) << 18 | z

    sK = pack(srcC)
    mK = pack(mC)
    mK_sorted = torch.sort(mK)[0]

    idx = torch.searchsorted(mK_sorted, sK)
    idx = torch.clamp(idx, 0, max(mK_sorted.numel() - 1, 0))

    keep = (mK_sorted[idx] == sK) if mK_sorted.numel() else torch.zeros_like(sK, dtype=torch.bool)

    if keep.sum().item() == 0:
        return None

    return ME.SparseTensor(features=src.F[keep], coordinates=src.C[keep], device=device)

def seen_lidar(lidar_vox, depth_vox, seen_map, T_cur, T_prev):
    if seen_map is not None and T_prev is not None:
        seen_map = warp_and_revoxelize(
            prev_voxels=seen_map,
            T_prev=T_prev,
            T_cur=T_cur,
            time_idx=0,
            debug=False,
            squash_offsets=False,
            depth_input=None
        )
    seen_map=union_unique(depth_vox,seen_map)
    lidar_seen = filter_to_mask(lidar_vox, seen_map)
    return lidar_seen, seen_map    

def create_empty_map():
    prev_coords=torch.cartesian_prod(torch.tensor([1]),torch.arange(-GRID_SIZE//2,GRID_SIZE//2),torch.arange(-GRID_SIZE//2,GRID_SIZE//2),torch.tensor([int(-0.35/VOXEL_SIZE)]),torch.tensor([1]))
    prev_feats=torch.full((prev_coords.shape[0],3),0.5) ; prev_feats[:,2] = 0
    return ME.SparseTensor(features=prev_feats, coordinates=prev_coords, device=device)

Evaluation metrics

In [ ]:
@torch.no_grad()
def chamfer_distance_eval(gt_pts:ME.SparseTensor, pred_pts:ME.SparseTensor, max_gt: int=1024, max_pred:int=1024):
    gt_pts, pred_pts = ME_to_points(gt_pts), ME_to_points(pred_pts)
    M,N=gt_pts.size(0),pred_pts.size(0)
    if N==0 or M==0:
        return torch.tensor(0.0,device=gt_pts.device)
    if M>max_gt:
        gt_pts=gt_pts[torch.randperm(M, device=gt_pts.device)[:max_gt]]
    if N>max_pred:
        pred_pts=pred_pts[torch.randperm(N, device=pred_pts.device)[:max_pred]]
    D = torch.cdist(gt_pts, pred_pts, p=2)
    return D.min(dim=1).values.mean() + D.min(dim=0).values.mean()

@torch.no_grad()
def volumetric_IoU(dense_gt, dense_pred):
    inter = (dense_gt&dense_pred).sum(dtype=torch.float32)
    union = (dense_gt|dense_pred).sum(dtype=torch.float32)
    return inter / union.clamp_min(1)

Training function and evaluation

In [ ]:
def train(depth_train,gt_train,pose_train,
          depth_val, gt_val, pose_val,
          num_epochs=NUM_EPOCHS,trunc=TRUNC, p=1, alpha=ALPHA,
          lr=LR,lrd=LRD,lrd_step=LRD_STEP,
          lambda_box=LAMBDA_BOX,lambda_reg=LAMBDA_REG,lambda_car=LAMBDA_CAR,lambda_rep=LAMBDA_REP,lambda_bce=LAMBDA_BCE, lambda_ray=LAMBDA_RAY,
          box_size=BOX_SIZE,rep_r=REP_R, pw=PW,
          m_dict=None):
    model = SparseUNet4D(prune_alpha=alpha).to(device)
    if m_dict:
        model.load_state_dict(m_dict["model_state"])
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    nb_seq_train,elapsed_seq = int(p*len(depth_train)),0
    depth_train, gt_train, pose_train= depth_train[:nb_seq_train], gt_train[:nb_seq_train], pose_train[:nb_seq_train]
    losses=[]
    eval = {"VIoU":[],"Chamfer":[],"F1":[], "Precision":[], "Recall":[]}
    mesh=o3d.io.read_triangle_mesh(f"{gt_train[0]}")
    mesh.compute_vertex_normals()
    gt=np.asarray(mesh.sample_points_uniformly(number_of_points=500000).points)
    del mesh

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        depth_train, gt_train, pose_train = shuffle(depth_train,gt_train,pose_train)
        epoch_loss=0.0
        model.train()

        for seq_idx in range(nb_seq_train):
            optimizer.zero_grad()
            output_voxels, T_prev, seen_map, prev_voxels = None, None, None, None
            loss, window_loss=0.0, 0.0
            elapsed_seq+=1
            if lrd_step!=0 and elapsed_seq%lrd_step==0:
                for pg in optimizer.param_groups:
                    pg["lr"]*=lrd

            for i in range(len(depth_train[seq_idx])):
                depth=np.loadtxt(f"{depth_train[seq_idx][i]}")
                T=load_pose(f"{pose_train[seq_idx][i]}")
                depth_input, gt_target= npy_to_voxel(depth,"synthetic_camera",time_idx=0), npy_to_voxel(world_to_robot(gt,T),"gt_mesh",time_idx=0)
                if output_voxels is not None:
                    prev_voxels = warp_and_revoxelize(output_voxels, T_prev, T, time_idx=1, squash_offsets=False)
                depth_input = union_unique(depth_input,prev_voxels)
                output_voxels, prun = model(depth_input)
                '''print("out.F  min/max/mean/std:",
                output_voxels.F.min().item(), output_voxels.F.max().item(),
                output_voxels.F.mean().item(), output_voxels.F.std().item())
                print("prun.F  min/max/mean/std:",
                prun[0].F.min().item(), prun[0].F.max().item(),
                prun[0].F.mean().item(), prun[0].F.std().item())'''

                loss=refined_loss(prun,output_voxels,gt_target,gt,np.linalg.inv(T_lidar_to_robot),device, 
                                    lambda_box=lambda_box,lambda_reg=lambda_reg,lambda_car=lambda_car,lambda_rep=lambda_rep,lambda_bce=lambda_bce, lambda_ray=lambda_ray,
                                    box_size=box_size,rep_r=rep_r, pw=pw)
                window_loss+=loss
                T_prev=T
                print(f"    Epoch {epoch+1}/{num_epochs}, Sequence {seq_idx+1}/{nb_seq_train}, Frame {i+1}/{SEQUENCE_LENGTH}, loss {loss}")

                if (i+1)%trunc==0 or i+1==len(depth_train[seq_idx]):
                    window_loss.backward()
                    total = 0.0
                    count = 0
                    for n,p in model.named_parameters():
                        if p.grad is None: 
                            continue
                        g = p.grad.detach().abs().mean().item()
                        total += g; count += 1
                    print("mean|grad| =", total/max(count,1))
                    optimizer.step()
                    epoch_loss += window_loss.item()
                    optimizer.zero_grad()
                    window_loss = 0.0
                    if output_voxels is not None:
                        output_voxels = ME.SparseTensor(features=output_voxels.F.detach(),coordinates=output_voxels.C,device=output_voxels.F.device)
                        nb1=output_voxels.C.shape[0]
                        output_voxels=suppress_by_pruning(output_voxels,prun[0],alpha)
                        print(f"Pruned: {(100*(nb1-output_voxels.C.shape[0])/nb1):.3f}%")
        losses.append(epoch_loss)

        model.eval()
        with torch.inference_mode():
            f1_metric = torchmetrics.classification.BinaryF1Score().to(device)
            precision_metric = torchmetrics.classification.BinaryPrecision().to(device)
            recall_metric = torchmetrics.classification.BinaryRecall().to(device)
            VIoU, cd, f1, prec, rec = 0, 0, 0, 0, 0
            for seq_idx in range(len(depth_val)):
                output_voxels=None
                for i in range(len(depth_val[seq_idx])):
                    depth=np.loadtxt(f"{depth_val[seq_idx][i]}")
                    T=load_pose(f"{pose_val[seq_idx][i]}")
                    depth_input, gt_target= npy_to_voxel(depth,"synthetic_camera",time_idx=0), npy_to_voxel(world_to_robot(gt,T),"gt_mesh",time_idx=0)
                    if output_voxels is not None:
                        prev_voxels = warp_and_revoxelize(output_voxels, T_prev, T, time_idx=1, squash_offsets=False)
                    depth_input = union_unique(depth_input,prev_voxels)
                    output_voxels, prun = model(depth_input)
                    T_prev=T
                    if (i+1)%trunc==0 and (i+1)>50:
                        output_voxels=suppress_by_pruning(output_voxels,prun[0],alpha)

                    if i>50:
                        dense_gt, dense_output = coords_to_dense(gt_target),coords_to_dense(output_voxels)
                        VIoU += float(volumetric_IoU(dense_gt,dense_output))
                        cd += float(chamfer_distance_eval(gt_target,output_voxels))
                        f1 += float(f1_metric(dense_gt, dense_output))
                        prec += float(precision_metric(dense_gt, dense_output))
                        rec += float(recall_metric(dense_gt,dense_output))
            eval["VIoU"].append(VIoU/((SEQUENCE_LENGTH-50)*len(depth_val))), eval["Chamfer"].append(cd/((SEQUENCE_LENGTH-50)*len(depth_val))), eval["F1"].append(f1/((SEQUENCE_LENGTH-50)*len(depth_val))), eval["Precision"].append(prec/((SEQUENCE_LENGTH-50)*len(depth_val))), eval["Recall"].append(rec/((SEQUENCE_LENGTH-50)*len(depth_val)))
        del depth_input, gt_target, output_voxels, prun
        gc.collect()
        torch.cuda.empty_cache()
    return model, losses, eval

Tuning

In [ ]:
'''model_dict = torch.load("unet4d.pt", map_location=device)'''

In [ ]:
best_tr_loss = float("inf")
best_config, m = None, None
losses, checkpoint, metrics = [], [], []

idx=1
for a in [0.2]:
    for b in [1]:
        for l in [0.001]:
            for ld in [0.96]:
                for t in [3,20]:
                    for l_bce in [15]:
                        for pw in [1]:
                            print(f"Config {idx}")
                            model,tr_loss,metrics=train(depth_train,gt_train,pose_train,depth_val,gt_val,pose_val,p=1,m_dict=None,alpha=a, box_size=b,lr=l,lrd=ld,trunc=t,lambda_bce=l_bce, pw=pw)
                            losses.append(tr_loss)
                            checkpoint.append( {
                                "model_state": deepcopy(model.state_dict()),
                                "config": {
                                    "alpha":a, "trunc": t,
                                    "lr":l, "decay": ld, "decay_step":LRD_STEP,
                                    "box_size":b, "rep_r":REP_R,
                                    "lambda_box":1,"lambda_reg":LAMBDA_REG,"lambda_car":LAMBDA_CAR,"lambda_rep":LAMBDA_REP,"lambda_bce":l_bce,
                                },
                                "losses": tr_loss,
                                "metrics": {
                                    "VIoU":deepcopy(metrics["VIoU"]),
                                    "Chamfer":deepcopy(metrics["Chamfer"]),
                                    "F1":deepcopy(metrics["F1"]),
                                    "Precision":deepcopy(metrics["Precision"]),
                                    "Recall":deepcopy(metrics["Recall"])
                                }
                            })
                            idx+=1
                        

torch.save(checkpoint,"unet4d.pt")

Plots

In [ ]:
for i in range(len(losses)):
    x,y = zip(*enumerate(losses[i]))
    al, bx, lr, ld, t, l_bce = checkpoint[i]['config']['alpha'], checkpoint[i]['config']['box_size'], checkpoint[i]['config']['lr'], checkpoint[i]['config']['decay'], checkpoint[i]['config']['trunc'], checkpoint[i]['config']["lambda_bce"]
    print(f"{i+1} - al:{al}, bx:{bx}, lr:{lr}, lrd:{ld}, t:{t}, l_bce: {l_bce}: loss {y[-1]}")
    plt.plot(x,y, label=f"{i+1}")
    plt.grid(True)
    plt.legend()
plt.show()
for metric in checkpoint[0]["metrics"]:
    plt.figure(figsize=(10, 5))
    best_max_idx = 0
    best_min_idx = 0

    for i, run in enumerate(checkpoint):
        raw_vals = run["metrics"][metric]
        vals = [v.item() if torch.is_tensor(v) else v for v in raw_vals]
        
        plt.plot(vals, label=f"Run {i+1}")
        
        # Logic to track best/worst performers based on last value
        if vals[-1] > [v.item() if torch.is_tensor(v) else v for v in checkpoint[best_max_idx]["metrics"][metric]][-1]:
            best_max_idx = i
        if vals[-1] < [v.item() if torch.is_tensor(v) else v for v in checkpoint[best_min_idx]["metrics"][metric]][-1]:
            best_min_idx = i

    plt.title(f"Metric: {metric}")
    plt.grid(True)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left') # Legend outside plot
    plt.tight_layout()
    plt.show()
    
    print(f"--- {metric} ---")
    print(f"Best Performance: Run {best_max_idx + 1}")
    print(f"Worst Performance: Run {best_min_idx + 1}\n")

Save

In [ ]:
torch.save(checkpoint[1],"unet4d.pt")

Load

In [ ]:
model_dict = torch.load("unet4d0.pt", map_location=device)
cfg = model_dict["config"]
model = SparseUNet4D(prune_alpha=cfg["alpha"]).to(device)
model.load_state_dict(model_dict["model_state"])

Testing

In [ ]:
def model_pass(model,depth_input,T_prev,T_cur,prev_voxels=None,prun=True):
    depth_input=npy_to_voxel(np.loadtxt(depth_input),"synthetic_depth",time_idx=0)
    if prev_voxels is not None:
        prev_voxels=warp_and_revoxelize(prev_voxels,T_prev,T_cur,1)
        depth_input = union_unique(depth_input,prev_voxels)
    output_voxels, pruning_scores = model(depth_input)
    if prun:
        output_voxels=suppress_by_pruning(output_voxels,pruning_scores[0],ALPHA)
    return output_voxels

with torch.no_grad():
    model.eval()
    for seq_idx in range(len(gt_val)):
        output_voxels,T_prev=None,None
        idx=0
        f1_metric = torchmetrics.classification.BinaryF1Score().to(device)
        precision_metric = torchmetrics.classification.BinaryPrecision().to(device)
        recall_metric = torchmetrics.classification.BinaryRecall().to(device)
        m={"VIoU":[], "cd":[], "f1":[], "prec":[], "rec":[]}

        mesh=o3d.io.read_triangle_mesh(f"{gt_train[seq_idx]}")
        mesh.compute_vertex_normals()
        gt=np.asarray(mesh.sample_points_uniformly(number_of_points=500000).points)

        for depth_frame, pose in zip(depth_val[seq_idx],pose_val[seq_idx]):
            idx+=1
            T_cur=load_pose(pose)
            output_voxels=model_pass(model,depth_frame,T_prev,T_cur,output_voxels,idx%1==0)
            T_prev=T_cur
            gt_target=world_to_robot(gt,T_cur)
            gt_target=npy_to_voxel(gt_target,False)
            if idx>50:
                dense_gt, dense_output = coords_to_dense(gt_target),coords_to_dense(output_voxels)
                m["VIoU"].append(float(volumetric_IoU(dense_gt,dense_output)))
                m["cd"].append(float(chamfer_distance_eval(gt_target,output_voxels)))
                m["f1"].append(float(f1_metric(dense_gt, dense_output)))
                m["prec"].append(float(precision_metric(dense_gt, dense_output)))
                m["rec"].append(float(recall_metric(dense_gt,dense_output)))
                
            '''output_points=voxel_to_npy(output_voxels)
            visualize_alignment(output_points,gt,nb_frames=10000)'''
        for met in m:
            plt.plot(m[met],label=f"{met} {seq_idx+1}")
            plt.legend()
            plt.grid(True)
        plt.show()
        output_points=voxel_to_npy(output_voxels)
        visualize_alignment(output_points,voxel_to_npy(gt_target),nb_frames=10000)